# Stage 1 - Fact extraction, full BanglaCHQ-Summ corpus

Decomposes every gold summary into atomic, typed facts. These are the targets the deletion and alteration pipelines act on in the **source** document.

Facts come from the **summary**, not the question. We need to corrupt in the source exactly what the summary claims; extracting from the source would target things the summary never mentions.

All three splits: train, valid, test. A `split` column is carried through so the benchmark can be partitioned later.

Output mirrors `dataset/NER/test_tag_columns.csv`: one column per category, facts within a column joined by ` | `.

**Cost controls.** Reasoning off, `max_tokens` capped, checkpoint every 50 rows with resume, and a pre-flight of one call. Projected: 2,350 calls, about $1.08.

In [1]:
import os
import sys
import threading
import time
from concurrent.futures import ThreadPoolExecutor

import pandas as pd
import requests
from dotenv import load_dotenv

sys.path.insert(0, os.path.abspath("source_corruption"))
from common import EXTRACT_FACTS_PROMPT, FACT_CATEGORIES, parse_facts

load_dotenv(os.path.abspath("../../.env"))
api_keys = [k for k in (os.getenv("OPENROUTER_API_KEY_NEW"), os.getenv("OPENROUTER_API_KEY")) if k]
if not api_keys:
    raise ValueError("No API key found! Check the .env file at the repository root.")

MODEL = "openai/gpt-5.6-luna"
SPLITS = ("train", "valid", "test")
BACKUP_FILE = "backup_facts_full.csv"
OUTPUT_LONG = "facts_full_long.csv"
OUTPUT_TAGS = "facts_full_tag_columns.csv"
CHECKPOINT_EVERY = 50
MAX_WORKERS = 6
MAX_TOKENS = 1024
FACT_SEP = "|||"
CAT_SEP = "::"

print(f"{len(api_keys)} key(s). Model: {MODEL}")

2 key(s). Model: openai/gpt-5.6-luna


In [2]:
frames = []
for split in SPLITS:
    f = pd.read_csv(f"../../BanglaCHQ-Summ/Dataset/{split}.csv")
    f["split"] = split
    frames.append(f)
df = pd.concat(frames, ignore_index=True)
df["question"] = df["question"].astype(str).str.strip()
df["summary"] = df["summary"].astype(str).str.strip()

assert df["id"].is_unique, "document ids are not unique across splits"
assert (df["question"].str.len() > 0).all() and (df["summary"].str.len() > 0).all(), "blank row"

print(df.groupby("split").size().to_string())
print("")
print(f"{len(df)} documents -> {len(df)} extraction calls")
print(f"summary length: mean {df['summary'].str.split().str.len().mean():.0f} words")

split
test      235
train    1880
valid     235

2350 documents -> 2350 extraction calls
summary length: mean 29 words


In [3]:
key_lock = threading.Lock()
active_key_index = 0


def call_model(prompt, max_retries=3):
    global active_key_index
    # Reasoning off - extraction is mechanical and reasoning tokens bill at the
    # completion rate. max_tokens is a cap, not a charge; 1024 gives headroom
    # for a long fact list and insures against truncation.
    payload = {"model": MODEL, "messages": [{"role": "user", "content": prompt}],
               "temperature": 0.0, "max_tokens": MAX_TOKENS,
               "reasoning": {"enabled": False}}
    for attempt in range(max_retries):
        for _ in range(len(api_keys)):
            with key_lock:
                k = api_keys[active_key_index]
            try:
                r = requests.post(
                    "https://openrouter.ai/api/v1/chat/completions",
                    headers={"Authorization": f"Bearer {k}", "Content-Type": "application/json"},
                    json=payload, timeout=180)
            except requests.RequestException:
                break
            if r.status_code == 200:
                body = r.json()
                return (body["choices"][0]["message"]["content"],
                        (body.get("usage") or {}).get("cost", 0.0))
            if r.status_code in (401, 402, 403, 429):
                with key_lock:
                    active_key_index = (active_key_index + 1) % len(api_keys)
                continue
            break
        time.sleep(2 * (attempt + 1))
    return None, 0.0


def run_one(row):
    raw, cost = call_model(EXTRACT_FACTS_PROMPT.format(summary=row["summary"]))
    facts = parse_facts(raw) if raw else []
    packed = FACT_SEP.join(f"{c}{CAT_SEP}{t}" for _, c, t in facts)
    return {"id": row["id"], "split": row["split"], "source": row["question"],
            "summary": row["summary"], "n_facts": len(facts), "cost": cost,
            "facts_raw": packed, "ok": bool(facts)}


jobs = [row for _, row in df.iterrows()]
print(f"{len(jobs)} calls queued.")

2350 calls queued.


In [4]:
# PRE-FLIGHT. One call before committing to 2,350.
probe = run_one(jobs[0])
print(f"facts={probe['n_facts']}  cost=${probe['cost']:.5f}")
for part in probe["facts_raw"].split(FACT_SEP):
    if part:
        cat, text = part.split(CAT_SEP, 1)
        print(f"   [{cat}] {text}")

if not probe["ok"]:
    raise RuntimeError("Pre-flight failed. Do not run the full extraction.")
print("")
print(f"Pre-flight passed. Projected total: ${probe['cost'] * len(jobs):.2f}")

facts=8  cost=$0.00030
   [Age] রোগীর বয়স ২৫ বছর।
   [Symptom] রোগীর ঘন ঘন টয়লেট হচ্ছে।
   [Symptom] রোগীর পেটে ব্যথা হচ্ছে।
   [Symptom] রোগীর টয়লেট পরিষ্কারভাবে হচ্ছে না।
   [Temporal] রোগীর ঠান্ডা লাগা, সর্দি ও হাঁচি দুই দিন যাবত হচ্ছে।
   [Symptom] রোগীর ঠান্ডা লেগেছে।
   [Symptom] রোগীর সর্দি হচ্ছে।
   [Symptom] রোগীর হাঁচি হচ্ছে।

Pre-flight passed. Projected total: $0.71


In [5]:
done = {}
if os.path.exists(BACKUP_FILE):
    prior = pd.read_csv(BACKUP_FILE)
    done = {r["id"]: r for _, r in prior.iterrows()}
    print(f"Resuming: {len(done)} documents already extracted.")
else:
    print("No backup - starting fresh.")

pending = [j for j in jobs if j["id"] not in done]
print(f"{len(pending)} of {len(jobs)} still to extract.")
print("")

start = time.time()
results = [done[j["id"]].to_dict() for j in jobs if j["id"] in done]
lock = threading.Lock()


def checkpoint():
    pd.DataFrame(results).to_csv(BACKUP_FILE, index=False, encoding="utf-8-sig")


if pending:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as pool:
        for finished, record in enumerate(pool.map(run_one, pending), start=1):
            with lock:
                results.append(record)
                if finished % CHECKPOINT_EVERY == 0 or finished == len(pending):
                    checkpoint()
                    ok = sum(bool(r.get("ok")) for r in results)
                    spent = sum(float(r.get("cost") or 0) for r in results)
                    print(f"   --- saved at {finished}/{len(pending)} "
                          f"(total {len(results)}, ok={ok}, ${spent:.3f}, "
                          f"{time.time() - start:.0f}s) ---")

checkpoint()
out = pd.DataFrame(results)
out["ok"] = out["ok"].astype(bool)
print("")
print(f"{len(out)} documents, {int(out['ok'].sum())} extracted, "
      f"{int((~out['ok']).sum())} failed, ${out['cost'].astype(float).sum():.3f}")

No backup - starting fresh.
2350 of 2350 still to extract.



   --- saved at 50/2350 (total 50, ok=50, $0.013, 39s) ---


   --- saved at 100/2350 (total 100, ok=99, $0.026, 44s) ---


   --- saved at 150/2350 (total 150, ok=149, $0.039, 65s) ---


   --- saved at 200/2350 (total 200, ok=199, $0.052, 84s) ---


...

   --- saved at 2150/2350 (total 2150, ok=2131, $0.565, 839s) ---


   --- saved at 2200/2350 (total 2200, ok=2181, $0.580, 858s) ---


   --- saved at 2250/2350 (total 2250, ok=2231, $0.592, 876s) ---


   --- saved at 2300/2350 (total 2300, ok=2281, $0.606, 894s) ---


   --- saved at 2350/2350 (total 2350, ok=2331, $0.619, 912s) ---

2350 documents, 2331 extracted, 19 failed, $0.619


In [6]:
# Long form: one row per fact.
records = []
for _, r in out[out["ok"]].iterrows():
    for i, part in enumerate(str(r["facts_raw"]).split(FACT_SEP), start=1):
        if not part:
            continue
        cat, text = part.split(CAT_SEP, 1)
        records.append({"id": r["id"], "split": r["split"], "fact_index": i,
                        "category": cat if cat in FACT_CATEGORIES else "Other",
                        "fact": text, "source": r["source"], "summary": r["summary"]})
facts_long = pd.DataFrame(records)
facts_long.to_csv(OUTPUT_LONG, index=False, encoding="utf-8-sig")
print(f"{len(facts_long)} facts -> {OUTPUT_LONG}")

# Tag-column form, matching dataset/NER/test_tag_columns.csv.
pivot = (facts_long.sort_values(["id", "fact_index"])
         .groupby(["id", "category"])["fact"].apply(lambda v: " | ".join(v))
         .unstack("category"))
tags = (out[out["ok"]][["id", "split", "source", "summary"]]
        .merge(pivot, on="id", how="left"))
for c in FACT_CATEGORIES:
    if c not in tags.columns:
        tags[c] = pd.NA
tags = tags[["id", "split", "source", "summary"] + FACT_CATEGORIES]
tags["n_facts"] = tags["id"].map(facts_long.groupby("id").size()).fillna(0).astype(int)
tags.to_csv(OUTPUT_TAGS, index=False, encoding="utf-8-sig")
print(f"{tags.shape} -> {OUTPUT_TAGS}")

13199 facts -> facts_full_long.csv
(2331, 15) -> facts_full_tag_columns.csv


In [7]:
print("facts per summary:")
print(tags["n_facts"].describe()[["mean", "min", "50%", "max"]].round(1).to_string())
print("")
print("by split:")
print(tags.groupby("split")["n_facts"].agg(["size", "mean"]).round(1).to_string())
print("")
print("facts by category, pilot share in brackets for comparison:")
pilot = {"Symptom": 47, "Temporal": 12, "Medicine": 11, "Age": 9, "Health Condition": 7,
         "Dosage": 5, "Test Result": 4, "Other": 3, "Medical Procedure": 2, "Specialist": 0}
counts = facts_long["category"].value_counts()
for c in FACT_CATEGORIES:
    n = int(counts.get(c, 0))
    print(f"   {c:<20} {n:6d}  {100 * n / len(facts_long):5.1f}%   [pilot {pilot.get(c, 0)}%]")

facts per summary:
mean     5.7
min      1.0
50%      5.0
max     22.0

by split:
       size  mean
split            
test    235   5.8
train  1862   5.6
valid   234   5.9

facts by category, pilot share in brackets for comparison:
   Symptom                6436   48.8%   [pilot 47%]
   Health Condition        745    5.6%   [pilot 7%]
   Medicine               1348   10.2%   [pilot 11%]
   Specialist               31    0.2%   [pilot 0%]
   Age                    1055    8.0%   [pilot 9%]
   Dosage                  257    1.9%   [pilot 5%]
   Medical Procedure       256    1.9%   [pilot 2%]
   Test Result             675    5.1%   [pilot 4%]
   Temporal               1749   13.3%   [pilot 12%]
   Other                   647    4.9%   [pilot 3%]
